**Step 1: Identify the Task**

*Task*: Guide a self-driving trailer across Kenya’s highway network (Nairobi, Central Highlands, Rift Valley, and Lake Victoria Basin regions) from Nairobi to Kisumu using both distance-based search knowledge and domain routing rules.

*Goal*: Determine the lowest-distance (km) valid route while accounting for highway connectivity, search-tree paths, and operational/road-condition rules.

**Step 2: Assemble Relevant Knowledge**

*Prior Exercise Artifacts*:

- Directed highway graph with exact Google Maps / search distances (km) across 28 towns/nodes (e.g., Nairobi -> Ruaka at 15.5 km, Ruaka -> Limuru at 21.0 km, Nakuru -> Kericho at 107.0 km, Kericho -> Ahero at 60.9 km, Ahero -> Kisumu at 23.5 km, yielding the baseline optimal path of 357.0 km).

- Unfolded search tree expanding all distinct paths from Nairobi to Kisumu without converging cycles.

*Retrieved Domain & Corridor Knowledge (APIs / Web Scraping / Official Sources)*:

- Kenya National Highways Authority (KeNHA) corridor classifications and regional road management data (such as the Northern Corridor A8/A104 and B1/A1 links across Rift Valley and Nyanza/Lake Victoria Basin).
- Practical routing context: standard A104/B1 corridors via Nakuru and Kericho are paved major highways, whereas alternative routes (via Eldoret/Kakamega or via Narok/Bomet/Kisii/Awach) represent longer or distinct regional detours.
- Data Acquisition Note (APIs & Scraping): In a full implementation script/notebook (e.g., fetch_kb_data.py), you can pull live or batch distances via a routing/distance API (such as Google Distance Matrix / Routes API or OpenRouteService) and scrape or parse KeNHA corridor advisories/notices into a JSON/CSV knowledge store before loading into the KB.

**Step 3: Define the Vocabulary**

*Constants (Entities & Values)*: Towns (Nairobi, Ruaka, Limuru, Naivasha, Gilgil, Nakuru, Kericho, Ahero, Kisumu, Eldoret, Narok, etc.), status values (Active, Blocked, Caution, Paved), and numeric distances (km).

*Predicates / Relations (Represented in Python):*

- Road(u, v, dist_km): Directed highway edge from town u to town v with distance dist_km.

- OptimalRoute(start, goal, path, cost_km): Derived shortest distance path and total cost from start to goal.

- RoadStatus(u, v, status): Current operational condition of edge (u, v).

- TrailerRule(condition, action): Domain rule governing trailer routing decisions.

In [ ]:
import heapq

# Step 4: Encode General Knowledge (Graph, Baseline UCS Reference, and Domain Rules)
highway_graph = {
    'Nairobi': {'Gitaru': 22.3, 'Ruaka': 15.5, 'Kiambu': 15.5, 'Ruiru': 24.6, 'Maai-Mahiu': 53.8},
    'Gitaru': {'Limuru': 20.1},
    'Ruaka': {'Limuru': 21.0},
    'Kiambu': {'Limuru': 25.7},
    'Ruiru': {'Thika': 21.4},
    'Maai-Mahiu': {'Narok': 89.1},
    'Limuru': {'Naivasha': 56.1},
    'Naivasha': {'Gilgil': 30.4},
    'Gilgil': {'Nakuru': 42.6},
    'Nakuru': {'Timboroa': 94.5, 'Eldama-Ravine': 63.1, 'Kericho': 107.0},
    'Timboroa': {'Eldoret': 61.7},
    'Eldama-Ravine': {'Eldoret': 108.0},
    'Eldoret': {'Kapsabet': 46.0, 'Kakamega': 106.0},
    'Kapsabet': {'Kisumu': 80.3},
    'Kakamega': {'Kisumu': 53.4},
    'Thika': {'Sagana': 63.9, 'Othaya': 86.8},
    'Sagana': {'Nyahururu': 156.0},
    'Othaya': {'Nyeri': 16.9},
    'Nyeri': {'Nyahururu': 104.0},
    'Nyahururu': {'Nakuru': 63.3},
    'Kericho': {'Ahero': 60.9},
    'Narok': {'Bomet': 81.0},
    'Bomet': {'Kisii': 82.4},
    'Kisii': {'Homa-Bay': 52.0, 'Awach': 77.4},
    'Homa-Bay': {'Ahero': 84.5},
    'Awach': {'Ahero': 14.1},
    'Ahero': {'Kisumu': 23.5},
    'Kisumu': {}
}

#Encoded corridor & operational facts retrieved for the KB
corridor_metadata = {
    "authority": "Kenya National Highways Authority (KeNHA)",
    "corridors_covered": ["Northern Corridor / A8 / A104", "Rift Valley & Lake Victoria Basin links"],
    "baseline_optimal_path": [
        "Nairobi", "Ruaka", "Limuru", "Naivasha", "Gilgil",
        "Nakuru", "Kericho", "Ahero", "Kisumu"
    ],
    "baseline_optimal_cost_km": 357.0
}

road_conditions = {}  # Key: (src, dst), Value: {"status": "Blocked" | "Caution", "reason": str, "penalty_km": float}

trailer_rules = {
    "Blocked": "Exclude segment from candidate search paths; reroute via next lowest-cost path.",
    "Caution": "Keep segment in search but flag extra trailer braking/stability caution.",
    "Default": "Traverse segment using shortest cumulative distance (Uniform Cost Search)."
}
#Step 5: Add New Facts to the Knowledge Base
def show_kb():
    print("\n--- Current Trailer Routing Knowledge Base ---")
    print(f"Total Origin Towns: {len(highway_graph)} | Corridor Authority: {corridor_metadata['authority']}")
    print("Active Road Condition Facts:")
    if not road_conditions:
        print("  (None — all graph edges operating at nominal distance cost)")
    for (u, v), info in road_conditions.items():
        print(f"  {u} -> {v} : {info['status']} (+{info.get('penalty_km', 0)} km) | {info['reason']}")
    print("Domain Rules:")
    for rule_key, action in trailer_rules.items():
        print(f"  {rule_key:8} -> {action}")
    print("--------------------------------------------")

def add_road_fact(src, dst, status, reason="Reported update", penalty_km=0.0):
    """Programmatically add or update a road condition fact in the KB."""
    src_t, dst_t, status_t = src.strip(), dst.strip(), status.strip().capitalize()
    road_conditions[(src_t, dst_t)] = {
        "status": status_t,
        "reason": reason,
        "penalty_km": float(penalty_km)
    }
    print(f"Added fact: Road({src_t} -> {dst_t}) = {status_t} ({reason})")

def add_fact_interactive():
    """Interactively add a new road status fact to the KB."""
    src = input("Enter origin town: ").strip()
    dst = input("Enter destination town: ").strip()
    status = input("Enter status (Blocked / Caution / Clear): ").strip().capitalize()
    reason = input("Enter reason/note: ").strip()
    penalty = input("Enter extra distance penalty in km (0 if none): ").strip()
    penalty_val = float(penalty) if penalty else 0.0
    if status == "Clear":
        road_conditions.pop((src, dst), None)
        print(f"Cleared override for {src} -> {dst}.")
    else:
        add_road_fact(src, dst, status, reason, penalty_val)

# Example programmatic fact additions
add_road_fact("Nakuru", "Kericho", "Caution", "Escarpment / highland mist & heavy trailer load", 0.0)
show_kb()

In [ ]:
#Step 5: Add New Facts to the Knowledge Base
def show_kb():
    print("\n--- Current Trailer Routing Knowledge Base ---")
    print(f"Total Origin Towns: {len(highway_graph)} | Corridor Authority: {corridor_metadata['authority']}")
    print("Active Road Condition Facts:")
    if not road_conditions:
        print("  (None — all graph edges operating at nominal distance cost)")
    for (u, v), info in road_conditions.items():
        print(f"  {u} -> {v} : {info['status']} (+{info.get('penalty_km', 0)} km) | {info['reason']}")
    print("Domain Rules:")
    for rule_key, action in trailer_rules.items():
        print(f"  {rule_key:8} -> {action}")
    print("--------------------------------------------")

def add_road_fact(src, dst, status, reason="Reported update", penalty_km=0.0):
    """Programmatically add or update a road condition fact in the KB."""
    src_t, dst_t, status_t = src.strip(), dst.strip(), status.strip().capitalize()
    road_conditions[(src_t, dst_t)] = {
        "status": status_t,
        "reason": reason,
        "penalty_km": float(penalty_km)
    }
    print(f"Added fact: Road({src_t} -> {dst_t}) = {status_t} ({reason})")

def add_fact_interactive():
    """Interactively add a new road status fact to the KB."""
    src = input("Enter origin town: ").strip()
    dst = input("Enter destination town: ").strip()
    status = input("Enter status (Blocked / Caution / Clear): ").strip().capitalize()
    reason = input("Enter reason/note: ").strip()
    penalty = input("Enter extra distance penalty in km (0 if none): ").strip()
    penalty_val = float(penalty) if penalty else 0.0
    if status == "Clear":
        road_conditions.pop((src, dst), None)
        print(f"Cleared override for {src} -> {dst}.")
    else:
        add_road_fact(src, dst, status, reason, penalty_val)

# Example programmatic fact additions
add_road_fact("Nakuru", "Kericho", "Caution", "Escarpment / highland mist & heavy trailer load", 0.0)
show_kb()